# DeepGalaxy: rendimiento e impacto temporal de la E/S

Este notebook procesa los CSV originales de DeepGalaxy, calcula métricas de rendimiento, escalabilidad e índice compuesto orientado al impacto temporal de la E/S.

La salida se separa en dos bloques:

- `01_individual_family/`: normalización por familia, usada para actualizar los análisis individuales de NFS y Lustre SC=1.
- `02_global_striping/`: normalización global por modo de acceso, usada para comparar SC=1, SC=2 y SC=4 en la técnica de *striping*.

Los nombres de los reportes y las figuras incorporan el sistema de ficheros, el modo de acceso, el `stripe count` y el tipo de normalización aplicado.


## Definición de las métricas y del índice compuesto

El impacto temporal real de la E/S se reporta mediante:

$$
\gamma = \frac{T_{io}}{T_{run}}
$$

donde \(T_{io}\) representa el tiempo de E/S y \(T_{run}\) el tiempo total de ejecución. Para las tablas de tesis, \(\gamma\) puede expresarse en porcentaje como:

$$
\gamma(\%) = \frac{T_{io}}{T_{run}}\times 100
$$

Como los valores absolutos de \(\gamma\) suelen ser pequeños, para el radar se utiliza una forma normalizada que permite comparar configuraciones dentro del grupo de normalización:

$$
\Gamma_{gain} = 1 - \frac{\gamma}{\max(\gamma)}
$$

De esta forma, todas las variables del radar tienen la misma dirección interpretativa: valores más altos indican mejor comportamiento. En particular, un valor alto de \(\Gamma_{gain}\) representa menor impacto relativo de la E/S sobre el tiempo total dentro del conjunto comparado.

La ganancia temporal de E/S se calcula como:

$$
Tio_{gain} = \frac{\min(T_{io})}{T_{io}}
$$

Por tanto, configuraciones con menor tiempo de E/S reciben valores más altos.

El índice compuesto usado en este notebook es:

$$
HealthScore_{io}
=
10\cdot
\frac{
BW_{norm}
+
IOPS_{norm}
+
Eff_{io,norm}
+
Eff_{run,norm}
+
Tio_{gain}
+
\Gamma_{gain}
}{6}
$$

donde:

$$
BW_{norm}=\frac{BW}{\max(BW)}, \qquad
IOPS_{norm}=\frac{IOPS}{\max(IOPS)}
$$

$$
Eff_{io,norm}=Eff_{io}, \qquad
Eff_{run,norm}=Eff_{run}
$$

siempre acotadas al intervalo \([0,1]\).

### Criterio de normalización

Para los análisis individuales de NFS y Lustre SC=1 se utiliza normalización por familia. Para la técnica de *striping*, se utiliza normalización global por modo de acceso, de modo que las configuraciones SC=1, SC=2 y SC=4 sean comparables bajo una misma escala.


In [4]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
sys.path.append(r"C:\Users\EParraga\NoteBook\PHD_THESIS\Tools\DeepIO_Perf_HPC_v2")

workdir=r"Data\A8_2026\DeepGalaxy\bw512"

INPUT_FILES = [
    os.path.join(workdir, "NFS_snGPU_e1", "results/6_Analisis_IO_DeepGalaxy_Perf_DG_bw512_f3c5x64_nfs_SnGPU_e1_JSC25_dep_v2.csv"),
   os.path.join(workdir, "Lustre_1", "results/6_Analisis_IO_DeepGalaxy_Perf_DG_bw512_f3c5x64_lustre_1ost_JSC25_dep_v2.csv"),
    os.path.join(workdir, "Lustre_2", "results/6_Analisis_IO_DeepGalaxy_Perf_DG_bw512_f3c5x64_lustre_2ost_JSC25_dep_v2.csv"),
    os.path.join(workdir, "Lustre_4", "results/6_Analisis_IO_DeepGalaxy_Perf_DG_bw512_f3c5x64_lustre_4ost_JSC25_dep_v2.csv"),
]
OUTPUT_DIR = Path(workdir,'deepgalaxy_io_temporal_reports_v4')

EXCLUDE_P = (48,)  # Cambiar a () si se desea conservar 12N--48p


## Funciones del pipeline

In [8]:
from __future__ import annotations

import argparse
import json
import math
import re
import zipfile
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------------
# DeepGalaxy I/O tuning analysis pipeline
# - Per-family normalization for NFS and Lustre SC=1 sections.
# - Striping-global normalization for comparing SC=1, SC=2 and SC=4.
# - Health score focused on temporal impact of I/O tuning.
# -----------------------------------------------------------------------------

DEFAULT_INPUTS = [
    os.path.join(workdir, "NFS_snGPU_e1", "results/6_Analisis_IO_DeepGalaxy_Perf_DG_bw512_f3c5x64_nfs_SnGPU_e1_JSC25_dep_v2.csv"),
    os.path.join(workdir, "Lustre_1", "results/6_Analisis_IO_DeepGalaxy_Perf_DG_bw512_f3c5x64_lustre_1ost_JSC25_dep_v2.csv"),
    os.path.join(workdir, "Lustre_2", "results/6_Analisis_IO_DeepGalaxy_Perf_DG_bw512_f3c5x64_lustre_2ost_JSC25_dep_v2.csv"),
    os.path.join(workdir, "Lustre_4", "results/6_Analisis_IO_DeepGalaxy_Perf_DG_bw512_f3c5x64_lustre_4ost_JSC25_dep_v2.csv"),
]

ACCESS_LABELS = {
    0: "shared",
    1: "shared+reload+shuffle",
    9: "shared",
}


def safe_div(a, b):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    return a / b.replace({0: np.nan})


def norm_max(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    mx = s.max(skipna=True)
    if pd.isna(mx) or mx <= 0:
        return pd.Series(np.zeros(len(s)), index=s.index, dtype=float)
    return s / mx


def gain_lower_is_better(s: pd.Series) -> pd.Series:
    """Gain in [0,1] for metrics where lower values are better."""
    s = pd.to_numeric(s, errors="coerce")
    mn = s.min(skipna=True)
    if pd.isna(mn) or mn <= 0:
        return pd.Series(np.zeros(len(s)), index=s.index, dtype=float)
    return (mn / s).clip(lower=0, upper=1)


def sanitize_name(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^a-zA-Z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "unknown"


def read_inputs(paths: Iterable[str]) -> pd.DataFrame:
    frames = []
    for p in paths:
        path = Path(p)
        if not path.exists():
            print(f"[WARN] No existe: {path}")
            continue
        df = pd.read_csv(path)
        df.columns = [c.strip() for c in df.columns]
        df["source_file"] = path.name
        frames.append(df)
    if not frames:
        raise FileNotFoundError("No se pudo leer ningún CSV de entrada.")
    return pd.concat(frames, ignore_index=True, sort=False)


def standardize(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Filesystem_Type"] = out.get("Filesystem_Type", "unknown").astype(str).str.lower().str.strip()
    out["File_Format"] = out.get("File_Format", "unknown").astype(str).str.lower().str.strip()
    out["Access_Mode"] = pd.to_numeric(out.get("Access_Mode", np.nan), errors="coerce").astype("Int64")
    out["Access_Mode_Label"] = out["Access_Mode"].map(lambda x: ACCESS_LABELS.get(int(x), f"mode_{x}") if pd.notna(x) else "unknown")
    out["Scale_Type"] = out.get("Scale_Type", "unknown").astype(str).str.lower().str.strip()
    out["N"] = pd.to_numeric(out.get("Nodes", out.get("N", np.nan)), errors="coerce")
    out["P"] = pd.to_numeric(out.get("Processes IO", out.get("P", np.nan)), errors="coerce")
    out["Processes_per_Node"] = pd.to_numeric(out.get("Processes_per_Node", out["P"] / out["N"]), errors="coerce")
    out["Batch_Size"] = pd.to_numeric(out.get("Batch_Size", np.nan), errors="coerce")
    out["Stripe_Count"] = pd.to_numeric(out.get("LUSTRE_STRIPE_COUNT", out.get("Stripe_Count", 0)), errors="coerce").fillna(0).astype(int)
    out.loc[out["Filesystem_Type"].str.lower() != "lustre", "Stripe_Count"] = 0
    out["T_io"] = pd.to_numeric(out.get("IO_Time(s)", np.nan), errors="coerce")
    out["T_run"] = pd.to_numeric(out.get("Run_Time(s)", np.nan), errors="coerce")
    out["BW_MiB_s"] = pd.to_numeric(out.get("Bandwidth(MiB/s)", np.nan), errors="coerce")
    out["IOPS"] = pd.to_numeric(out.get("IOPs_DS", out.get("IOPs_FS", np.nan)), errors="coerce")
    out["D_app_bytes"] = pd.to_numeric(out.get("IO_Total(bytes)", np.nan), errors="coerce")
    out["Gamma"] = safe_div(out["T_io"], out["T_run"])
    out["cfg_label"] = out.apply(lambda r: f"{int(r['N'])}N--{int(r['P'])}p" if pd.notna(r["N"]) and pd.notna(r["P"]) else "unknown", axis=1)
    out["family"] = out.apply(
        lambda r: f"{r['Filesystem_Type']}|{r['File_Format']}|{r['Access_Mode_Label']}|SC{int(r['Stripe_Count'])}", axis=1
    )
    return out


def aggregate(df: pd.DataFrame, exclude_p: Iterable[int] = (48,)) -> pd.DataFrame:
    work = df.copy()
    if exclude_p:
        work = work[~work["P"].isin(list(exclude_p))].copy()
    keys = ["Filesystem_Type", "File_Format", "Access_Mode", "Access_Mode_Label", "Scale_Type", "Stripe_Count", "N", "P", "Processes_per_Node", "Batch_Size"]
    metrics = ["T_io", "T_run", "BW_MiB_s", "IOPS", "D_app_bytes", "Gamma"]
    agg = work.groupby(keys, dropna=False)[metrics].agg(["mean", "std", "count"]).reset_index()
    agg.columns = ["_".join(c).rstrip("_") if isinstance(c, tuple) else c for c in agg.columns]
    rename = {
        "T_io_mean": "T_io_mean", "T_run_mean": "T_run_mean", "BW_MiB_s_mean": "BW_mean", "IOPS_mean": "IOPS_mean",
        "D_app_bytes_mean": "D_app_bytes_mean", "Gamma_mean": "Gamma_mean", "T_io_count": "n_runs"
    }
    agg = agg.rename(columns=rename)
    agg["cfg_label"] = agg.apply(lambda r: f"{int(r['N'])}N--{int(r['P'])}p", axis=1)
    agg["family"] = agg.apply(lambda r: f"{r['Filesystem_Type']}|{r['File_Format']}|{r['Access_Mode_Label']}|SC{int(r['Stripe_Count'])}", axis=1)
    return agg.sort_values(["Filesystem_Type", "Access_Mode", "Stripe_Count", "P"]).reset_index(drop=True)


def add_scaling_metrics(agg: pd.DataFrame) -> pd.DataFrame:
    out = []
    for fam, g in agg.groupby("family", dropna=False):
        g = g.sort_values("P").copy()
        ref = g.iloc[0]
        p_ref = ref["P"]
        scale = g["P"] / p_ref
        g["Speedup_io"] = ref["T_io_mean"] / g["T_io_mean"]
        g["Eff_io"] = g["Speedup_io"] / scale
        g["Speedup_run"] = ref["T_run_mean"] / g["T_run_mean"]
        g["Eff_run"] = g["Speedup_run"] / scale
        g["Gamma"] = g["Gamma_mean"]
        out.append(g)
    return pd.concat(out, ignore_index=True)


def add_health_score(agg: pd.DataFrame, scope: str) -> pd.DataFrame:
    """
    scope = 'family'               -> independent normalization per family.
    scope = 'striping_global_mode' -> normalize Lustre SC=1/2/4 together, separately per access mode.
    scope = 'global'               -> normalize over all rows.
    """
    df = agg.copy()
    if scope == "family":
        group_cols = ["family"]
    elif scope == "striping_global_mode":
        group_cols = ["Filesystem_Type", "File_Format", "Access_Mode_Label"]
    elif scope == "global":
        group_cols = []
    else:
        raise ValueError(f"scope no reconocido: {scope}")

    if group_cols:
        groups = df.groupby(group_cols, dropna=False)
    else:
        groups = [("global", df)]

    pieces = []
    for _, g in groups:
        g = g.copy()
        g["BW_norm"] = norm_max(g["BW_mean"])
        g["IOPS_norm"] = norm_max(g["IOPS_mean"])
        g["Eff_io_norm"] = pd.to_numeric(g["Eff_io"], errors="coerce").clip(0, 1)
        g["Eff_run_norm"] = pd.to_numeric(g["Eff_run"], errors="coerce").clip(0, 1)
        g["Tio_gain"] = gain_lower_is_better(g["T_io_mean"])
        # Impacto relativo de E/S. Gamma se reporta como T_io/T_run.
        # Para el radar se usa una ganancia normalizada, de forma que valores altos
        # representen menor impacto relativo de E/S dentro de la familia/grupo analizado.
        gamma = pd.to_numeric(g["Gamma"], errors="coerce")
        gamma_max = gamma.max(skipna=True)
        if pd.isna(gamma_max) or gamma_max <= 0:
            g["Gamma_gain"] = pd.Series(np.ones(len(g)), index=g.index, dtype=float)
        else:
            g["Gamma_gain"] = (1.0 - (gamma / gamma_max)).clip(0, 1)
        components = ["BW_norm", "IOPS_norm", "Eff_io_norm", "Eff_run_norm", "Tio_gain", "Gamma_gain"]
        g["Health_Score_IO"] = 10.0 * g[components].mean(axis=1, skipna=True)
        g["normalization_scope"] = scope
        pieces.append(g)
    return pd.concat(pieces, ignore_index=True)


def make_latex_table(df: pd.DataFrame, caption: str, label: str, columns: list[str], formats: dict[str, str] | None = None) -> str:
    formats = formats or {}
    def fmt_val(col, val):
        if pd.isna(val):
            return "--"
        kind = formats.get(col, "2f")
        if kind == "int":
            return f"{int(round(float(val))):,}".replace(",", r"\,")
        if kind == "pct":
            return f"{100*float(val):.2f}"
        if kind == "3f":
            return f"{float(val):.3f}"
        return f"{float(val):.2f}"

    headers = {
        "cfg_label": r"\textbf{Config.}", "BW_mean": r"\textbf{BW (MiB/s)}", "IOPS_mean": r"\textbf{IOPS}",
        "T_io_mean": r"\textbf{$T_{io}$ (s)}", "T_run_mean": r"\textbf{$T_{run}$ (s)}",
        "Speedup_io": r"\textbf{$Speedup_{io}$}", "Eff_io": r"\textbf{$Eff_{io}$}", "Eff_run": r"\textbf{$Eff_{run}$}",
        "Gamma": r"\textbf{$\gamma$}", "BW_norm": r"\textbf{$BW_{norm}$}", "IOPS_norm": r"\textbf{$IOPS_{norm}$}",
        "Eff_io_norm": r"\textbf{$Eff_{io}$}", "Eff_run_norm": r"\textbf{$Eff_{run}$}", "Tio_gain": r"\textbf{$Tio_{gain}$}",
        "Gamma_gain": r"\textbf{$Gamma_{gain}$}", "Health_Score_IO": r"\textbf{Health Score}", "Stripe_Count": r"\textbf{SC}"
    }
    align = "l" + " c"*(len(columns)-1)
    lines = [r"\begin{table}[H]", r"\centering", f"\\caption{{{caption}}}", f"\\label{{{label}}}", r"\scriptsize", f"\\begin{{tabular}}{{{align}}}", r"\toprule"]
    lines.append(" & ".join(headers.get(c, c) for c in columns) + r" \\")
    lines.append(r"\midrule")
    for _, row in df.iterrows():
        vals = []
        for c in columns:
            if c == "cfg_label":
                vals.append(str(row[c]))
            elif c == "Stripe_Count":
                vals.append(str(int(row[c])))
            else:
                vals.append(fmt_val(c, row[c]))
        lines.append(" & ".join(vals) + r" \\")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}", ""])
    return "\n".join(lines)


def plot_radar(df: pd.DataFrame, title: str, outpath: Path, axes: list[str] | None = None):
    axes = axes or ["BW_norm", "IOPS_norm", "Eff_io_norm", "Eff_run_norm", "Tio_gain", "Gamma_gain"]
    labels = {
        "BW_norm": "Bandwidth\n(BW)",
        "IOPS_norm": "I/O Ops\n(IOPS)",
        "Eff_io_norm": "I/O efficiency\n(Eff_io)",
        "Eff_run_norm": "Run efficiency\n(Eff_run)",
        "Tio_gain": "I/O time gain\n(Tio_gain)",
        "Gamma_gain": "I/O impact gain\n(1-γ/γmax)",
    }
    data = df.sort_values("P").copy()
    n = len(axes)
    angles = np.linspace(0, 2*np.pi, n, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
    for _, row in data.iterrows():
        values = [float(row[a]) if pd.notna(row[a]) else 0.0 for a in axes]
        values += values[:1]
        label = f"{row['cfg_label']} | H={row['Health_Score_IO']:.2f}"
        ax.plot(angles, values, linewidth=2, label=label)
        ax.fill(angles, values, alpha=0.08)
    ax.set_theta_offset(np.pi/2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([labels.get(a, a) for a in axes], fontsize=10)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(["20%", "40%", "60%", "80%", "100%"], fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_title(title, pad=25, fontsize=13, fontweight="bold")
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.12), fontsize=8)
    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_health_bar(df: pd.DataFrame, title: str, outpath: Path):
    data = df.sort_values("P").copy()
    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = np.arange(len(data))
    ax.bar(x, data["Health_Score_IO"])
    ax.set_xticks(x)
    ax.set_xticklabels(data["cfg_label"], rotation=30, ha="right")
    ax.set_ylim(0, 10)
    ax.set_ylabel("Health Score (0--10)")
    ax.set_title(title, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    for i, v in enumerate(data["Health_Score_IO"]):
        ax.text(i, v + 0.12, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)


def generate_outputs(scored_family: pd.DataFrame, scored_striping: pd.DataFrame, output_dir: Path):
    """Generate reports with a strict separation between individual and global analyses.

    Directory layout:
    output_dir/
      01_individual_family/
        csv/
        latex/
        figures/radar/
        figures/health_bar/
      02_global_striping/
        csv/
        latex/
        figures/radar/
        figures/health_bar/
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    individual_dir = output_dir / "01_individual_family"
    global_dir = output_dir / "02_global_striping"

    ind_csv_dir = individual_dir / "csv"
    ind_latex_dir = individual_dir / "latex"
    ind_radar_dir = individual_dir / "figures" / "radar"
    ind_bar_dir = individual_dir / "figures" / "health_bar"

    glob_csv_dir = global_dir / "csv"
    glob_latex_dir = global_dir / "latex"
    glob_radar_dir = global_dir / "figures" / "radar"
    glob_bar_dir = global_dir / "figures" / "health_bar"

    for d in [ind_csv_dir, ind_latex_dir, ind_radar_dir, ind_bar_dir,
              glob_csv_dir, glob_latex_dir, glob_radar_dir, glob_bar_dir]:
        d.mkdir(parents=True, exist_ok=True)

    individual_latex_lines = []
    global_latex_lines = []

    perf_cols = ["cfg_label", "BW_mean", "IOPS_mean", "T_io_mean", "T_run_mean", "Speedup_io", "Eff_io", "Eff_run", "Gamma"]
    health_cols = ["cfg_label", "BW_norm", "IOPS_norm", "Eff_io_norm", "Eff_run_norm", "Tio_gain", "Gamma_gain", "Health_Score_IO"]

    def fs_title(fs: str) -> str:
        return "NFS" if str(fs).lower() == "nfs" else "Lustre"

    def sc_text(fs: str, sc: int) -> str:
        return "" if str(fs).lower() != "lustre" else f" SC={sc}"

    # ------------------------------------------------------------------
    # 1) Individual reports: per-family normalization.
    #    Purpose: update the already written analyses for NFS and Lustre SC=1
    #    without mixing their normalization scales with the striping study.
    # ------------------------------------------------------------------
    for fam, g in scored_family.groupby("family"):
        g = g.sort_values("P").copy()
        fs = g["Filesystem_Type"].iloc[0]
        mode = g["Access_Mode_Label"].iloc[0]
        sc = int(g["Stripe_Count"].iloc[0])
        fs_label = fs_title(fs)
        sc_label = sc_text(fs, sc)
        name = sanitize_name(f"deepgalaxy_{fs}_sc{sc}_{mode}_individual_family")

        g.to_csv(ind_csv_dir / f"{name}.csv", index=False)

        individual_latex_lines.append(make_latex_table(
            g,
            f"Rendimiento e impacto temporal de E/S para DeepGalaxy en {fs_label}{sc_label} bajo modo {mode} con normalización por familia.",
            f"tab:{name}_performance",
            perf_cols,
            {"IOPS_mean": "int", "Eff_io": "3f", "Eff_run": "3f", "Gamma": "3f"}
        ))
        individual_latex_lines.append(make_latex_table(
            g,
            f"Variables normalizadas e índice compuesto para DeepGalaxy en {fs_label}{sc_label} bajo modo {mode} con normalización por familia.",
            f"tab:{name}_health",
            health_cols,
            {}
        ))

        radar_title = (
            f"DEEPGALAXY I/O TEMPORAL HEALTH FINGERPRINT\n"
            f"Individual analysis | File system: {fs_label}{sc_label} | Format: HDF5\n"
            f"Data loading mode: {mode} | Normalization: per family"
        )
        bar_title = (
            f"Health Score IO - Individual analysis\n"
            f"{fs_label}{sc_label} | {mode} | Normalization: per family"
        )
        plot_radar(g, radar_title, ind_radar_dir / f"{name}_radar6.png")
        plot_health_bar(g, bar_title, ind_bar_dir / f"{name}_health_bar.png")

    scored_family.to_csv(ind_csv_dir / "deepgalaxy_individual_family_normalization_all.csv", index=False)
    (ind_latex_dir / "latex_tables_individual_family.tex").write_text("\n".join(individual_latex_lines), encoding="utf-8")

    # ------------------------------------------------------------------
    # 2) Global striping reports: common normalization for SC=1, SC=2 and SC=4
    #    within each access mode. Purpose: compare the striping technique fairly.
    # ------------------------------------------------------------------
    lustre = scored_striping[scored_striping["Filesystem_Type"].str.lower() == "lustre"].copy()
    for mode, gmode in lustre.groupby("Access_Mode_Label"):
        gmode = gmode.sort_values(["Stripe_Count", "P"]).copy()
        name = sanitize_name(f"deepgalaxy_lustre_striping_{mode}_global")
        gmode.to_csv(glob_csv_dir / f"{name}.csv", index=False)

        # Full global table across all scales and SC values.
        global_latex_lines.append(make_latex_table(
            gmode.assign(cfg_label=gmode.apply(lambda r: f"SC={int(r['Stripe_Count'])} | {r['cfg_label']}", axis=1)),
            f"Evolución global del rendimiento e impacto temporal de E/S para la técnica de striping en Lustre bajo modo {mode}.",
            f"tab:{name}_all_scales",
            perf_cols,
            {"IOPS_mean": "int", "Eff_io": "3f", "Eff_run": "3f", "Gamma": "3f"}
        ))

        maxp = gmode["P"].max()
        dec = gmode[gmode["P"] == maxp].sort_values("Stripe_Count").copy()
        global_latex_lines.append(make_latex_table(
            dec,
            f"Comparación global de striping en Lustre bajo modo {mode} para la máxima escala analizada.",
            f"tab:{name}_decision",
            ["Stripe_Count"] + perf_cols[1:] + ["Health_Score_IO"],
            {"IOPS_mean": "int", "Eff_io": "3f", "Eff_run": "3f", "Gamma": "3f"}
        ))
        global_latex_lines.append(make_latex_table(
            dec,
            f"Variables normalizadas de la comparación global de striping en Lustre bajo modo {mode}.",
            f"tab:{name}_health",
            ["Stripe_Count"] + health_cols[1:],
            {}
        ))

        # Radar per SC using global striping normalization.
        for sc, gsc in gmode.groupby("Stripe_Count"):
            radar_title = (
                f"DEEPGALAXY I/O TEMPORAL HEALTH FINGERPRINT\n"
                f"Global striping analysis | File system: Lustre SC={int(sc)} | Format: HDF5\n"
                f"Data loading mode: {mode} | Normalization: global across SC=1,2,4"
            )
            plot_radar(gsc, radar_title, glob_radar_dir / f"{name}_sc{int(sc)}_radar6.png")

        # Decision bar chart at maximum scale.
        dec_bar = dec.copy()
        dec_bar["cfg_label"] = dec_bar["Stripe_Count"].map(lambda x: f"SC={int(x)}")
        bar_title = (
            f"Health Score IO - Global striping comparison\n"
            f"Lustre | {mode} | {int(maxp)} processes | Normalization: global across SC=1,2,4"
        )
        plot_health_bar(dec_bar, bar_title, glob_bar_dir / f"{name}_decision_health_bar.png")

    scored_striping.to_csv(glob_csv_dir / "deepgalaxy_global_striping_normalization_all.csv", index=False)
    (glob_latex_dir / "latex_tables_global_striping.tex").write_text("\n".join(global_latex_lines), encoding="utf-8")

    # Global README / summary
    md = [
        "# DeepGalaxy performance and temporal I/O impact report",
        "",
        "## Output organization",
        "",
        "- `01_individual_family/`: análisis individuales con normalización por familia. Se usa para actualizar NFS y Lustre SC=1.",
        "- `02_global_striping/`: análisis comparativo de la técnica de striping con normalización global por modo de acceso.",
        "",
        "## Índice compuesto",
        r"$HealthScore_{io}=10\cdot\frac{BW_{norm}+IOPS_{norm}+Eff_{io}+Eff_{run}+Tio_{gain}+Gamma_{gain}}{6}$",
        "",
        "donde `Tio_gain` recompensa menor tiempo de E/S y `Gamma_gain=1-gamma/max(gamma)` recompensa menor impacto relativo de E/S dentro del grupo de normalización. Se reporta `gamma=Tio/Trun` como impacto temporal real.",
    ]
    (output_dir / "README_report.md").write_text("\n".join(md), encoding="utf-8")

    with zipfile.ZipFile(output_dir.with_suffix(".zip"), "w", compression=zipfile.ZIP_DEFLATED) as z:
        for f in output_dir.rglob("*"):
            z.write(f, f.relative_to(output_dir.parent))
    return output_dir.with_suffix(".zip")

def run_pipeline(input_files, output_dir="/data/deepgalaxy_io_temporal_reports_v4", exclude_p=(48,)):
    raw = read_inputs(input_files)
    std = standardize(raw)
    agg = aggregate(std, exclude_p=exclude_p)
    agg = add_scaling_metrics(agg)
    family = add_health_score(agg, scope="family")
    striping = add_health_score(agg, scope="striping_global_mode")
    outzip = generate_outputs(family, striping, Path(output_dir))
    return family, striping, outzip


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--inputs", nargs="*", default=DEFAULT_INPUTS)
    parser.add_argument("--output-dir", default="/data/deepgalaxy_io_temporal_reports_v4")
    parser.add_argument("--exclude-p", nargs="*", type=int, default=[48])
    args = parser.parse_args()
    family, striping, outzip = run_pipeline(args.inputs, args.output_dir, tuple(args.exclude_p))
    print(f"[OK] family rows={len(family)} striping rows={len(striping)}")
    print(f"[OK] ZIP: {outzip}")




## Ejecutar procesamiento

In [10]:
family_df, striping_df, zip_path = run_pipeline(INPUT_FILES, output_dir=str(OUTPUT_DIR), exclude_p=EXCLUDE_P)
print(zip_path)
print(f"family_df: {family_df.shape}")
print(f"striping_df: {striping_df.shape}")

Data\A8_2026\DeepGalaxy\bw512\deepgalaxy_io_temporal_reports_v4.zip
family_df: (40, 43)
striping_df: (40, 43)


## Vista rápida: normalización por familia

In [12]:
cols = ["Filesystem_Type", "Access_Mode_Label", "Stripe_Count", "cfg_label", "BW_mean", "IOPS_mean", "T_io_mean", "T_run_mean", "Eff_io", "Eff_run", "Gamma", "Health_Score_IO"]
family_df.sort_values(["Filesystem_Type", "Access_Mode_Label", "Stripe_Count", "P"])[cols].head(20)

,Filesystem_Type,Access_Mode_Label,Stripe_Count,cfg_label,BW_mean,IOPS_mean,T_io_mean,T_run_mean,Eff_io,Eff_run,Gamma,Health_Score_IO
15,lustre,shared,1,1N--4p,187.432005,4.812426e+04,11.212027,26808.028333,1.000000,1.000000,0.000419,6.987908
16,lustre,shared,1,2N--8p,243.770776,6.328138e+04,8.239657,14529.803975,0.680370,0.922519,0.000575,6.961013
17,lustre,shared,1,4N--16p,322.934486,8.563368e+04,6.284354,7159.264033,0.446029,0.936131,0.000883,7.356847
18,lustre,shared,1,8N--32p,366.496327,1.008725e+05,5.826076,4474.726350,0.240557,0.748873,0.001326,6.865372
19,lustre,shared,1,16N--64p,400.501885,1.175685e+05,5.795052,2792.903667,0.120922,0.599914,0.002078,6.201394
20,lustre,shared,2,1N--4p,159.203703,4.087648e+04,12.258807,31389.291900,1.000000,1.000000,0.000391,7.260672
21,lustre,shared,2,2N--8p,194.488816,5.048808e+04,10.094012,15406.660750,0.607232,1.018692,0.000658,7.002833
22,lustre,shared,2,4N--16p,240.846036,6.386365e+04,8.526112,7719.290533,0.359449,1.016586,0.001105,7.011942
23,lustre,shared,2,8N--32p,265.719069,7.313296e+04,8.009763,4636.050400,0.191310,0.846337,0.001790,6.448820
24,lustre,shared,2,16N--64p,325.874684,9.564959e+04,7.098142,2538.881600,0.107940,0.772715,0.002821,6.467758


## Vista rápida: normalización global para striping

In [15]:
striping_df.query("Filesystem_Type == 'lustre'").sort_values(["Access_Mode_Label", "Stripe_Count", "P"])[cols].head(30)

,Filesystem_Type,Access_Mode_Label,Stripe_Count,cfg_label,BW_mean,IOPS_mean,T_io_mean,T_run_mean,Eff_io,Eff_run,Gamma,Health_Score_IO
0,lustre,shared,1,1N--4p,187.432005,4.812426e+04,11.212027,26808.028333,1.000000,1.000000,0.000419,7.076297
1,lustre,shared,1,2N--8p,243.770776,6.328138e+04,8.239657,14529.803975,0.680370,0.922519,0.000575,7.082340
2,lustre,shared,1,4N--16p,322.934486,8.563368e+04,6.284354,7159.264033,0.446029,0.936131,0.000883,7.543282
3,lustre,shared,1,8N--32p,366.496327,1.008725e+05,5.826076,4474.726350,0.240557,0.748873,0.001326,7.145302
4,lustre,shared,1,16N--64p,400.501885,1.175685e+05,5.795052,2792.903667,0.120922,0.599914,0.002078,6.640181
5,lustre,shared,2,1N--4p,159.203703,4.087648e+04,12.258807,31389.291900,1.000000,1.000000,0.000391,6.798997
6,lustre,shared,2,2N--8p,194.488816,5.048808e+04,10.094012,15406.660750,0.607232,1.018692,0.000658,6.438313
7,lustre,shared,2,4N--16p,240.846036,6.386365e+04,8.526112,7719.290533,0.359449,1.016586,0.001105,6.320225
8,lustre,shared,2,8N--32p,265.719069,7.313296e+04,8.009763,4636.050400,0.191310,0.846337,0.001790,5.686867
9,lustre,shared,2,16N--64p,325.874684,9.564959e+04,7.098142,2538.881600,0.107940,0.772715,0.002821,5.540506


## Tablas de decisión para la técnica de striping

In [18]:
decision = []
for mode, g in striping_df.query("Filesystem_Type == 'lustre'").groupby("Access_Mode_Label"):
    maxp = g["P"].max()
    decision.append(g[g["P"] == maxp].sort_values("Stripe_Count"))
decision_df = pd.concat(decision, ignore_index=True)
decision_df[["Access_Mode_Label", "Stripe_Count", "cfg_label", "BW_mean", "IOPS_mean", "T_io_mean", "T_run_mean", "Eff_io", "Eff_run", "Gamma", "Health_Score_IO"]]

,Access_Mode_Label,Stripe_Count,cfg_label,BW_mean,IOPS_mean,T_io_mean,T_run_mean,Eff_io,Eff_run,Gamma,Health_Score_IO
0,shared,1,16N--64p,400.501885,1.175685e+05,5.795052,2792.903667,0.120922,0.599914,0.002078,6.640181
1,shared,2,16N--64p,325.874684,9.564959e+04,7.098142,2538.881600,0.107940,0.772715,0.002821,5.540506
2,shared,4,16N--64p,348.381760,1.022558e+05,6.591564,2789.800100,0.077086,0.643789,0.002392,5.819551
3,shared+reload+shuffle,1,16N--64p,9827.040012,2.500568e+06,16.355838,2591.076000,0.115128,0.674409,0.006323,6.864153
4,shared+reload+shuffle,2,16N--64p,6918.714851,1.760458e+06,23.016224,2923.264550,0.136735,0.618653,0.007863,5.066015
5,shared+reload+shuffle,4,16N--64p,6807.468023,1.732559e+06,24.305551,2583.747200,0.139746,0.685074,0.009423,4.805568


## Archivos generados

La salida queda organizada en `01_individual_family/` y `02_global_striping/`.

In [21]:
for p in sorted(OUTPUT_DIR.rglob("*")):
    print(p)

Data\A8_2026\DeepGalaxy\bw512\deepgalaxy_io_temporal_reports_v4\01_individual_family
Data\A8_2026\DeepGalaxy\bw512\deepgalaxy_io_temporal_reports_v4\01_individual_family\csv
Data\A8_2026\DeepGalaxy\bw512\deepgalaxy_io_temporal_reports_v4\01_individual_family\csv\deepgalaxy_individual_family_normalization_all.csv
Data\A8_2026\DeepGalaxy\bw512\deepgalaxy_io_temporal_reports_v4\01_individual_family\csv\deepgalaxy_lustre_sc1_shared_individual_family.csv
Data\A8_2026\DeepGalaxy\bw512\deepgalaxy_io_temporal_reports_v4\01_individual_family\csv\deepgalaxy_lustre_sc1_shared_reload_shuffle_individual_family.csv
Data\A8_2026\DeepGalaxy\bw512\deepgalaxy_io_temporal_reports_v4\01_individual_family\csv\deepgalaxy_lustre_sc2_shared_individual_family.csv
Data\A8_2026\DeepGalaxy\bw512\deepgalaxy_io_temporal_reports_v4\01_individual_family\csv\deepgalaxy_lustre_sc2_shared_reload_shuffle_individual_family.csv
Data\A8_2026\DeepGalaxy\bw512\deepgalaxy_io_temporal_reports_v4\01_individual_family\csv\deepgal